In [1]:
import pandas as pd,numpy as np
import matplotlib.pyplot as plt,seaborn as sns
import kagglehub

In [2]:
path = kagglehub.dataset_download("yasserh/loan-default-dataset")

df = pd.read_csv(path + "/Loan_Default.csv", encoding="latin1")

df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [3]:
df = df.drop(['year','ID'],axis=1)
NumCols = df.select_dtypes(exclude=object).columns
CatCols = df.select_dtypes(object).columns

In [4]:
NumCols

Index(['loan_amount', 'rate_of_interest', 'Interest_rate_spread',
       'Upfront_charges', 'term', 'property_value', 'income', 'Credit_Score',
       'LTV', 'Status', 'dtir1'],
      dtype='object')

In [5]:
df.isnull().sum()

loan_limit                    3344
Gender                           0
approv_in_adv                  908
loan_type                        0
loan_purpose                   134
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                            41
Neg_ammortization              121
interest_only                    0
lump_sum_payment                 0
property_value               15098
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                        9150
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                            200
submission_of_application      200
LTV                          15098
Region              

In [6]:
from src.transformers.Transformers import EnCoder,CatGroupModeImputer,NumGroupMeanImputer
from sklearn.pipeline import Pipeline

In [7]:
NumGrpImputeCols= ['Region','loan_type','business_or_commercial']
CatGrpImputeCols = ['Status','loan_type','Security_Type','credit_type','Gender','co-applicant_credit_type']

In [8]:
# NumtargetimputeCols =['income','property_value','term']
# CattargetimputeCols=['loan_limit','approv_in_adv','loan_purpose','Neg_ammortization','submission_of_application','age']

In [9]:
Selective_num_imputer = Pipeline([
    ('income',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='income')),
    ('property_value',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='property_value')),
    ('term',NumGroupMeanImputer(group_cols=NumGrpImputeCols,target_col='term'))
])



Selective_cat_imputer = Pipeline([
    ('loan_limit',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='loan_limit')),
    ('approv_in_adv',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='approv_in_adv')),
    ('loan_purpose',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='loan_purpose')),
    ('Neg_ammortization',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='Neg_ammortization')),
    ('submission_of_application',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='submission_of_application')),
    ('age',CatGroupModeImputer(group_cols=CatGrpImputeCols,target_col='age'))
])

In [10]:
SelectiveImputer = Pipeline([
    ('num',Selective_num_imputer),
    ('cat',Selective_cat_imputer)
])

In [11]:
newdf = SelectiveImputer.fit_transform(df,'y')
newdf.isnull().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                          15098
Region              

In [12]:
newdf['LTV'] = newdf['LTV'].fillna(newdf['loan_amount'] / newdf['property_value'])

In [13]:
newdf.isnull().sum()

loan_limit                       0
Gender                           0
approv_in_adv                    0
loan_type                        0
loan_purpose                     0
Credit_Worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest             36439
Interest_rate_spread         36639
Upfront_charges              39642
term                             0
Neg_ammortization                0
interest_only                    0
lump_sum_payment                 0
property_value                   0
construction_type                0
occupancy_type                   0
Secured_by                       0
total_units                      0
income                           0
credit_type                      0
Credit_Score                     0
co-applicant_credit_type         0
age                              0
submission_of_application        0
LTV                              0
Region              

In [13]:
y = newdf[['rate_of_interest','Interest_rate_spread','Upfront_charges']]
X = newdf.drop(['rate_of_interest','Interest_rate_spread','Upfront_charges','dtir1'],axis=1)

In [14]:
X.columns

Index(['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'term', 'Neg_ammortization', 'interest_only',
       'lump_sum_payment', 'property_value', 'construction_type',
       'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type',
       'Credit_Score', 'co-applicant_credit_type', 'age',
       'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status'],
      dtype='object')

In [19]:
from sklearn.ensemble import RandomForestRegressor

In [15]:
y_roi = y['rate_of_interest']

mask_roi = y_roi.notna()

X_train_roi = X[mask_roi]
y_train_roi = y_roi[mask_roi]

X_test_roi = X[~mask_roi]

In [16]:
X_test_roi

,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,term,...,income,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status
0,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,360.0,...,1740.00000,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1
1,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,360.0,...,4980.00000,EQUI,552,EXP,55-64,to_inst,0.677644,North,direct,1
10,cf,Male,nopre,type2,p3,l2,nopc,b/c,136500,300.0,...,4020.00000,EXP,723,CIB,55-64,to_inst,81.250000,North,direct,1
12,cf,Joint,nopre,type2,p3,l1,nopc,b/c,206500,360.0,...,3780.00000,CRIF,884,EXP,65-74,to_inst,80.038760,North,direct,1
15,cf,Male,nopre,type1,p4,l1,nopc,nob/c,76500,360.0,...,2220.00000,EXP,685,CIB,45-54,not_inst,55.434783,North,direct,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148651,cf,Joint,nopre,type3,p3,l1,nopc,nob/c,446500,360.0,...,12300.00000,EXP,897,EXP,45-54,to_inst,87.893701,North,direct,1
148652,cf,Male,nopre,type1,p1,l1,nopc,nob/c,96500,180.0,...,5460.00000,EQUI,608,EXP,55-64,to_inst,0.180158,North,direct,1
148658,cf,Sex Not Available,nopre,type1,p4,l1,nopc,nob/c,386500,360.0,...,4680.00000,EQUI,669,EXP,25-34,to_inst,0.691739,south,direct,1
148661,cf,Sex Not Available,nopre,type2,p4,l1,nopc,b/c,346500,360.0,...,4436.10806,EXP,585,CIB,25-34,to_inst,96.787710,south,direct,1


In [35]:
y_irs = y['Interest_rate_spread']

mask_irs = y_irs.notna()

X_train_irs = X[mask_irs]
y_train_irs = y_irs[mask_irs]

X_test_irs = X[~mask_irs]

In [36]:
y_uc = y['Upfront_charges']

mask_uc = y_uc.notna()

X_train_uc = X[mask_uc]
y_train_uc = y_uc[mask_uc]

X_test_uc = X[~mask_uc]

In [37]:
rf_roi = RandomForestRegressor()

In [38]:
rf_roi.fit(X_train_roi,y_test_roi)

ValueError: could not convert string to float: 'cf'

In [42]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

roi_cols = [
'Credit_Score','loan_amount','income','term','LTV',
'loan_type','loan_purpose','Region','construction_type','occupancy_type'
]

X_roi = pd.get_dummies(df[roi_cols], drop_first=True)
y_roi = df['rate_of_interest']

mask_roi = y_roi.notna()

X_train_roi = X_roi[mask_roi]
y_train_roi = y_roi[mask_roi]

X_test_roi = X_roi[~mask_roi]

model_roi = RandomForestRegressor(n_estimators=300, random_state=42)

model_roi.fit(X_train_roi, y_train_roi)

pred_roi = model_roi.predict(X_test_roi)

df.loc[X_test_roi.index,'rate_of_interest'] = pred_roi


KeyboardInterrupt

